# Gate B（Colab）：P+K（Parser→Kev，Sys1）vs Gemma 4 E2B（純 LLM）意圖解析模組比較

**只比意圖解析階段（模組層）**，不做 Full path。同一批合成金標（`gate_data.py`，seed 7，cal/test 切分），P+K 門檻只用 cal 決定，報告只看 test。

| 臂 | 內容 |
|---|---|
| P+K | 原 Parser＋`parser_v2` 修正層 → Kev-0.8B（`jaredpalmer/kev-0.8b@v1.0`＝revision `9a45d25e`，與伺服器同版）只答 noul／choice，JSON 由程式組出 |
| Gemma `q8_0` / `q4_0` | `gemma-4-E2B-it`，llama.cpp：8-bit＝GGUF Q8_0（代替 SFP8，見下）、4-bit＝Google 官方 QAT Q4_0 |
| 每種量化 × 8 臂 | schema `ext`／`min`／`legacy`（＋`min_fs` few-shot）× 解碼 `prompt`（只靠 prompt）／`json`（JSON-schema 文法受限解碼） |

**執行方式**：執行階段選 **T4 GPU**，然後依序執行全部儲存格。中斷或斷線後**從頭再執行一次**即可：建置有快取，收集會從斷點續跑（結果在 Drive 上）。
預估時間（T4）：第一次建置 llama.cpp 約 30–60 分鐘（只有第一次；之後從 Drive 快取複製，幾秒鐘）、Kev 收集幾分鐘、每種量化 8 臂約 30–60 分鐘。

**v2（2026-10-19 第一次結果之後）**：同一個 `RUN_NAME` 重跑全部儲存格即可補齊——Kev 會用新增的「急症」題重問全部句子（約 1–2 分鐘）；Gemma 只重收輸出含 thinking 的列（主要是 q8_0 的 json 臂）；另加 30 句安全補充集（`extra_safety.py`，只評安全）。舊列會保留為 `.thinking-leak.bak`。

**v3（production 原文 prompt）**：新增 `prod_json` 臂（production 的 SYSTEM_PROMPT、INTENT_JSON_SCHEMA、user 包裝原文，取自 source bundle），報告另外用 production 程式碼重播決定性層，算出「Production 現行路徑」。同一個 `RUN_NAME` 重跑即可，只會補收 prod_json。

**v2 加跑**：Kev-4B（fp16，看 P+K 上限）與 Gemma 4 12B Q4_0（4 臂，和 TAIDE-12B 同大小）。任何一個模型起不來會記在「失敗清單」並繼續跑其他模型。12B 能否被這版 llama.cpp 載入、Kev-4B 在 fp16 下數值是否正常，都是第一次實測。

**v4（hold-out：router_gold）**：設定格的 `DATASET = "router_gold"`（預設）會跑第二份資料集——300 句、事前凍結門檻與評分規則（`holdout/SCORING.md`）。會先核對 `holdout/FREEZE.json`，報告第一行標示 FROZEN ✓／✗。Kev 0.8B、4B 和三個 Gemma 都要跑，不跑併發階梯，約 1–1.5 小時。**只跑一次**；中斷就重跑全部儲存格（會續跑），不要換 `RUN_NAME`。要回到 gate_data 就把 `DATASET` 改回 `"gate_data"`。

**限制（報告會標註）**：金標是合成資料；延遲只看相對倍率與輸出 token 數；Colab 的併發與排隊數字不可移植；Gemma 都是量化模型（非原始精度）。
SFP8 是 gemma.cpp 的格式，llama.cpp 不能載入，所以 8-bit 臂用 Q8_0。T4 不支援 BF16，所以 Kev 用 fp32（Kev 評估用的就是 fp32 路徑）。

In [ ]:
# ===== 設定（通常只需要改 RUN_NAME）=====
DRIVE_ROOT  = "/content/drive/MyDrive/gateB"           # 結果、快取、bundle 都在這裡
BUNDLE_ZIP  = f"{DRIVE_ROOT}/gateB-colab.zip"          # 先把 gateB-colab.zip 上傳到這個位置（或下一格會請你上傳）
DATASET     = "router_gold"                             # "router_gold" = 凍結的 hold-out（v4）；"gate_data" = 原本的 cal/test
RUN_NAME    = "gateB-holdout-001" if DATASET == "router_gold" else "gateB-t4-001"   # 同名 = 續跑
ARMS        = "ext_json,ext_prompt,min_json,min_prompt,legacy_json,legacy_prompt,min_fs_json,min_fs_prompt,prod_json"
ARMS_12B    = "min_json,min_fs_prompt,ext_prompt,legacy_prompt,prod_json"   # 12B 每句慢 3–4 倍，只跑 5 臂
# prod_json＝production 原文 SYSTEM_PROMPT＋INTENT_JSON_SCHEMA（strict）＋production 的 user 包裝、max_output_tokens 400（vendor_prod/）
SKIP_KEV    = True                                      # Kev 的資料已收齊；只補 Gemma 的 prod_json 時設 True，跳過階段 1
RUN_LATENCY = True                                      # 併發階梯；只供形狀參考
# Kev：(標籤, 權重, dtype)。0.8b＝production 候選（和伺服器同版）；4b＝看 P+K 上限（T4 上 fp32 放不下，用 fp16）
KEV_MODELS  = [("0.8b", "jaredpalmer/kev-0.8b@v1.0", "auto"),
               ("4b",   "jaredpalmer/kev-4b@v1.0",   "fp16")]
# Gemma：(標籤, 臂, llama-server 參數, 併發階梯的臂, 併發等級)。標籤沒有大小前綴＝E2B
# production prompt 約 4–6k token，每個 slot 至少要 8k：E2B 用 2 個 slot×8k，12B 用 1 個 slot×12k（併發階梯沿用先前結果）
GEMMA_MODELS = [("q8_0",     ARMS,     "--parallel 2 --ctx 16384", "min_json,ext_json", "1,2,4"),
                ("q4_0",     ARMS,     "--parallel 2 --ctx 16384", "min_json,ext_json", "1,2,4"),
                ("12b-q4_0", ARMS_12B, "--parallel 1 --ctx 12288", "min_json",          "1,2")]
REDO_LATENCY = set()                                    # 併發階梯都已量過，沿用
if DATASET == "router_gold":                            # 臂與模型是事前登錄的（SCORING.md §3），不要改
    SKIP_KEV, RUN_LATENCY = False, False
    GEMMA_MODELS = [("q4_0",     "min_json,min_fs_prompt,prod_json", "--parallel 2 --ctx 16384", "", ""),
                    ("q8_0",     "min_json,prod_json",               "--parallel 2 --ctx 16384", "", ""),
                    ("12b-q4_0", "min_fs_prompt,prod_json",          "--parallel 1 --ctx 12288", "", "")]
MODEL_DIR   = "/content/models"                         # GGUF 放本機磁碟（每次 session 重新下載約 1–2 分鐘，不佔 Drive 空間）

CODE  = "/content/gateB-colab"
RUN   = f"{DRIVE_ROOT}/runs/{RUN_NAME}"
STATE = "/content/gateb_state"
import os; os.environ.update(GATEB_STATE=STATE, GATEB_CACHE=f"{DRIVE_ROOT}/cache", HF_HUB_ENABLE_HF_TRANSFER="0")
try:                                                    # 選用：Colab「密鑰」裡的 HF_TOKEN（這些 repo 是 Apache-2.0，通常不需要）
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception: pass

In [ ]:
# ===== 掛載 Drive、解開 bundle =====
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists(BUNDLE_ZIP):
    from google.colab import files
    print("找不到", BUNDLE_ZIP, "→ 請選擇 gateB-colab.zip 上傳")
    up = files.upload(); name = next(iter(up)); shutil.copy(name, BUNDLE_ZIP)
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall("/content")
assert os.path.exists(f"{CODE}/gateb.py"), "zip 內應有 gateB-colab/ 資料夾"
!cd {CODE}/vendor_fk && sha256sum -c SHA256SUMS --quiet && echo "vendor_fk 與 fullpath-kev 原檔一致"
!nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv

In [ ]:
# ===== 先用假伺服器跑一次自我測試（幾秒，不用 GPU）=====
!cd {CODE} && bash selftest.sh

In [ ]:
# ===== 安裝：llama.cpp（CUDA，快取在 Drive）＋ Kev（固定 commit、獨立 uv venv）=====
# 第一次要編譯 llama.cpp 的 CUDA 版，T4 的 2 顆 vCPU 約 30–60 分鐘，每分鐘印一次進度；之後的 session 直接用 Drive 快取。
# 想縮短第一次建置：把下一行改成 LLAMA_CUDA_FA=OFF（不編 FlashAttention kernel，約快一半，只影響 Gemma 的絕對延遲）。
LLAMA_CUDA_FA = "ON"
!cd {CODE} && LLAMA_CUDA_FA={LLAMA_CUDA_FA} bash colab_setup.sh 2>&1 | tee {STATE}-setup.log

In [ ]:
# ===== 共用小工具 =====
import subprocess, json, sys, time, shlex
def sh(cmd, ok=(0,)):
    """串流輸出；回傳 exit code。130 = 被中斷（部分結果已寫入，重跑這格會續跑）"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout: print(line, end="")
        p.wait()
    except KeyboardInterrupt:
        p.send_signal(2); p.wait(); print("\n[中斷] 部分結果已保存；重跑這格即可續跑"); raise
    if p.returncode not in ok and p.returncode != 130: raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p.returncode
PY = f"cd {CODE} && python3 -I"
G  = f"{PY} gateb.py --dataset {DATASET} --run {shlex.quote(RUN)}"
SV = f"{PY} servers.py --state {STATE}"
os.makedirs(STATE, exist_ok=True)
kev_commit = subprocess.run("git -C /content/kev rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
llama_ver  = subprocess.run("/content/llama-bin/llama-server --version 2>&1 | tail -2", shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " ")
sh(f'{G} env --set kev_commit={kev_commit} --set "llama_commit={llama_ver}" --set kev_model={KEV_MODELS[0][1]}')
FAILED = []                                             # 某個模型起不來時記在這裡，其餘照跑
if DATASET == "router_gold":                            # 凍結核對：任何凍結檔案被改過就停下來
    ok = subprocess.run(f"{PY} holdout.py", shell=True, capture_output=True, text=True)
    print(ok.stdout.strip() or ok.stderr.strip())
    assert ok.returncode == 0, "凍結檔案與 FREEZE.json 不符——請重新解開原始 gateB-colab.zip，不要改程式"

## 階段 1：P+K（Kev 單獨占用 GPU；0.8B → 4B 依序）

In [ ]:
cc = float(subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.split()[0])
for lab, model, dtype in ([] if SKIP_KEV else KEV_MODELS):
    if dtype == "auto": dtype = "bf16" if cc >= 8.0 else "fp32"
    print(f"\n===== Kev {lab}: {model} ({dtype}) =====")
    try:
        sh(f"{SV} kev start --kev-dir /content/kev --model {model} --dtype {dtype}")
        served = subprocess.run(f"grep -a '^serving' {STATE}/kev.log | tail -1", shell=True, capture_output=True, text=True).stdout.strip()
        print(served)
        sfx = "" if lab == "0.8b" else f"_{lab}"
        sh(f'{G} env --set kev_dtype{sfx}={dtype} --set "kev_served{sfx}={served}"')
        sh(f"{G} kev-collect --kev-label {lab} --kev-url http://127.0.0.1:8230")
        if RUN_LATENCY: sh(f"{G} latency --target kev --kev-label {lab} --kev-url http://127.0.0.1:8230 --levels 1,2,4 --n 48 {'--redo' if lab in REDO_LATENCY else ''}")
    except KeyboardInterrupt:
        raise
    except Exception as e:
        FAILED.append(f"Kev {lab}: {e}"); print(f"[略過] Kev {lab} 失敗：{e}（log：{STATE}/kev.log）")
    finally:
        sh(f"{SV} kev stop")

## 階段 2：Gemma（每個模型各起一次 llama-server：E2B Q8 → E2B Q4 → 12B Q4）

In [ ]:
for q, arms, srv_args, lat_arms, lat_levels in GEMMA_MODELS:
    print(f"\n===== Gemma {q} =====")
    try:
        r = subprocess.run(f"{PY} fetch_gguf.py --quant {q} --dest {MODEL_DIR}", shell=True, capture_output=True, text=True)
        if r.returncode: raise RuntimeError(f"下載失敗：{(r.stderr or r.stdout).strip()[-500:]}")
        info = json.loads(r.stdout.strip().splitlines()[-1])
        print(q, "→", info["repo"], info["file"], info["revision"][:12], f"{info['bytes']/2**30:.2f} GiB")
        sh(f'{G} env --set gguf_{q}={info["repo"]}/{info["file"]}@{info["revision"][:12]}')
        sh(f"{SV} llama start --bin /content/llama-bin/llama-server --gguf {shlex.quote(info['path'])} {srv_args}")
        sh(f"{G} gemma-collect --quant {q} --arms {arms} --redo-thinking --llm-url http://127.0.0.1:8350")
        if RUN_LATENCY: sh(f"{G} latency --target gemma --quant {q} --arms {lat_arms} --levels {lat_levels} --n 48 --llm-url http://127.0.0.1:8350 {'--redo' if q in REDO_LATENCY else ''}")
    except KeyboardInterrupt:
        raise
    except Exception as e:
        FAILED.append(f"Gemma {q}: {e}"); print(f"[略過] Gemma {q} 失敗：{e}（log：{STATE}/llama.log）")
    finally:
        sh(f"{SV} llama stop")
print("\n失敗清單：", FAILED or "無")

## 結果

In [ ]:
sh(f"{G} analyze"); sh(f"{G} status")
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN}/{'HOLDOUT_SUMMARY.md' if DATASET == 'router_gold' else 'SUMMARY.md'}", encoding="utf-8").read()))

In [ ]:
# ===== 打包結果（Drive 上一份 zip）→ 帶回伺服器 =====
import shutil
z = shutil.make_archive(f"{DRIVE_ROOT}/results-{RUN_NAME}", "zip", root_dir=f"{DRIVE_ROOT}/runs", base_dir=RUN_NAME)
print("結果 zip：", z)
print(f"""帶回伺服器（使用者規定的路徑）：
  mkdir -p /var/lib/jenkins/tomod/ops/vllm-experiment-suite/gateB-colab/results
  unzip results-{RUN_NAME}.zip -d /var/lib/jenkins/tomod/ops/vllm-experiment-suite/gateB-colab/results/
在伺服器上可離線重算（不需 GPU）：
  cd /var/lib/jenkins/tomod/ops/vllm-experiment-suite/gateB-colab && python3 gateb.py --dataset {DATASET} analyze --run results/{RUN_NAME}""")

### 疑難排解
- **斷線／中斷**：重新執行全部儲存格。llama.cpp 用 Drive 快取不重建；Kev／Gemma 收集都從 jsonl 斷點續跑；`SUMMARY.md` 每一步都會重寫（標示 PARTIAL）。
- **Kev 起不來**：看 `/content/gateb_state/kev.log`。CUDA graph 有問題時，把階段 1 的 `kev start` 加上 `--cuda-graphs 0`。
- **llama-server 起不來**：看 `/content/gateb_state/llama.log`。記憶體不足時把 `--parallel 4 --ctx 16384` 改成 `--parallel 1 --ctx 4096`（併發階梯就只剩 VU=1 有意義）。
- **想換成 production 的 intent prompt**：改 `prompts/legacy_system.txt`（欄位名稱不變），換一個 `RUN_NAME` 重跑（prompt 簽章不同，舊列不要混用）。
- **埠**：Kev 只用 127.0.0.1:8230、llama-server 只用 127.0.0.1:8350；8000/8001/8002/8003/8010 一律拒絕。